# No split strategy

The entire fileset is passed to the executor as a single unit.
Coffea's `Runner` internally distributes files across workers,
but the whole run is one job from coffea-workflow's perspective.

**Trade-off:** lowest overhead, but if the run fails everything must restart from scratch — no partial results are cached. In this example the whole analysis breaks, as the first file in the fileset is broken.

In [1]:
import sys
sys.path.insert(0, "..")

from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run
from coffea_workflow import facilities
from analysis import get_fileset, run_analysis, plot_results

In [2]:
step_fileset = Step(name="Fileset", step_type=Fileset, builder=get_fileset)
step_analysis = Step(name="Analysis", step_type=Analysis, builder=run_analysis)
step_plotting = Step(name="Plotting", step_type=Plotting, builder=plot_results)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

Step(name='Plotting', step_type=<class 'coffea_workflow.artifacts.Plotting'>, builder=<function plot_results at 0x7f7b3d3cec00>, builder_params=None, facility=None, executor_config=None, input=None, output=None)

In [3]:
config = RunConfig(
    strategy=None,
    cache_dir=".cache_no_split",
    facility=facilities.local,
    executor_config=ExecutorConfig(executor_type="FuturesExecutor"),
)

run(workflow, config)

Workflow DAG:
  [0] Fileset -> Fileset builder=<function get_fileset at 0x7f7b3d3ce660>
  [1] Analysis -> Analysis builder=<function run_analysis at 0x7f7b3d3ce7a0>
  [2] Plotting -> Plotting builder=<function plot_results at 0x7f7b3d3cec00>
Edges:
  Fileset -> Analysis
  Analysis -> Plotting
Executing step 'Fileset' of type 'Fileset' with the user code <function get_fileset at 0x7f7b3d3ce660> and user parameters None
Extracted from cache: .cache_no_split/Fileset/22c2cc185640e94941a58e99ac76a71eff60afb8890cd48945c40208ad50cd89
  -> materialized at .cache_no_split/Fileset/22c2cc185640e94941a58e99ac76a71eff60afb8890cd48945c40208ad50cd89
Executing step 'Analysis' of type 'Analysis' with the user code <function run_analysis at 0x7f7b3d3ce7a0> and user parameters None
Extracted from cache: .cache_no_split/Chunking/5d7233ce64f30f5aea5d8fab5f3b987fdf3d37f2d81c69f32befaa463464a0ac

No split strategy was specified, proceed with processing the whole fileset...
-----------------------------------

Output()

/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1278: UserWarning: Performed attempt 1 out of 4
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1278: UserWarning: Performed attempt 2 out of 4
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1278: UserWarning: Performed attempt 3 out of 4
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1278: UserWarning: Performed attempt 4 out of 4
  warnings.warn(


loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/loky/process_executor.py", line 490, in _process_worker
    r = call_item()
        ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/loky/process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py", line 1290, in automatic_retries
    raise e
  File "/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py", line 1276, in automatic_retries
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py", line 1360, in metadata_fetcher_root
    with uproot.open(
         ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/uproot/reading.py", line 144, in open
    file = ReadOnlyFile(
           ^^^^^^

Failure caught!
  -> materialized at .cache_no_split/Analysis/af829d369589e92f54ed6fb97c33479dfb18a10803721c01befc641c6a8bc24f
Executing step 'Plotting' of type 'Plotting' with the user code <function plot_results at 0x7f7b3d3cec00> and user parameters None


TypeError: 'NoneType' object is not subscriptable